# Vehicle Dataset Loading

This notebook reads the four vehicle dataset CSV files into pandas DataFrames.

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
base_path = Path("vehicle-dataset-from-cardekho")

file_1 = base_path / "car data.csv"
file_2 = base_path / "CAR DETAILS FROM CAR DEKHO.csv"
file_3 = base_path / "Car details v3.csv"
file_4 = base_path / "car details v4.csv"

df1 = pd.read_csv(file_1)
df2 = pd.read_csv(file_2)
df3 = pd.read_csv(file_3)
df4 = pd.read_csv(file_4)

print("All files loaded successfully.")

In [ ]:
print("df1 shape:", df1.shape)
print("df2 shape:", df2.shape)
print("df3 shape:", df3.shape)
print("df4 shape:", df4.shape)

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df3.head()

In [ ]:
df4.head()

## Merge The Four Files

The files have different column names, so we first rename similar columns, then combine them into one dataframe.

In [ ]:
df1_renamed = df1.rename(columns={
    "Car_Name": "name",
    "Selling_Price": "selling_price",
    "Present_Price": "present_price",
    "Kms_Driven": "km_driven",
    "Fuel_Type": "fuel",
    "Seller_Type": "seller_type",
    "Transmission": "transmission",
    "Owner": "owner"
})

df4_renamed = df4.rename(columns={
    "Make": "make",
    "Model": "name",
    "Price": "selling_price",
    "Year": "year",
    "Kilometer": "km_driven",
    "Fuel Type": "fuel",
    "Transmission": "transmission",
    "Owner": "owner",
    "Seller Type": "seller_type",
    "Engine": "engine",
    "Max Power": "max_power",
    "Max Torque": "torque",
    "Seating Capacity": "seats"
})

In [ ]:
all_cars = pd.concat([
    df1_renamed,
    df2,
    df3,
    df4_renamed
], ignore_index=True, sort=False)

print("Merged shape:", all_cars.shape)
all_cars.head()

In [ ]:
all_cars.columns

## Histogram And Box Plot For Numeric Columns

This section uses seaborn with a black theme to visualize each numeric column in the merged dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style="darkgrid")
sns.set_palette("Greys")

plt.rcParams["figure.facecolor"] = "black"
plt.rcParams["axes.facecolor"] = "black"
plt.rcParams["savefig.facecolor"] = "black"
plt.rcParams["axes.edgecolor"] = "white"
plt.rcParams["axes.labelcolor"] = "white"
plt.rcParams["xtick.color"] = "white"
plt.rcParams["ytick.color"] = "white"
plt.rcParams["text.color"] = "white"
plt.rcParams["grid.color"] = "gray"

In [ ]:
numeric_columns = all_cars.select_dtypes(include="number").columns
numeric_columns

In [ ]:
for column in numeric_columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("black")

    sns.histplot(all_cars[column].dropna(), kde=True, ax=axes[0], color="white")
    axes[0].set_title(f"Histogram of {column}", color="white")
    axes[0].set_xlabel(column, color="white")
    axes[0].set_ylabel("Frequency", color="white")

    sns.boxplot(x=all_cars[column].dropna(), ax=axes[1], color="dimgray")
    axes[1].set_title(f"Box Plot of {column}", color="white")
    axes[1].set_xlabel(column, color="white")

    plt.tight_layout()
    plt.show()

## Data Preparation For Machine Learning

This part starts from the merged dataframe, cleans the columns, prepares features and target, and finishes with `train_test_split`.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df = all_cars.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df = df.drop_duplicates()
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Convert columns that contain numbers stored as text into numeric values.
numeric_text_columns = ["mileage", "engine", "max_power", "torque", "seats", "present_price", "km_driven", "year", "length", "width", "height", "fuel_tank_capacity"]

for column in numeric_text_columns:
    if column in df.columns:
        df[column] = df[column].astype(str).str.extract(r"([0-9]+\.?[0-9]*)")[0]
        df[column] = pd.to_numeric(df[column], errors="coerce")

df.head()

In [ ]:
# Create a car age feature.
if "year" in df.columns:
    df["car_age"] = 2026 - df["year"]

In [ ]:
target_column = "selling_price"

df = df.dropna(subset=[target_column])

for column in df.columns:
    if column != target_column:
        if df[column].dtype == "object":
            df[column] = df[column].fillna(df[column].mode()[0] if not df[column].mode().empty else "Unknown")
        else:
            df[column] = df[column].fillna(df[column].median())

In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column]

X = pd.get_dummies(X, drop_first=True)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)